# Kiwi — reconocer herramientas desde el celular

EmbeddingGemma 2 + Gradio: una página para subir una foto y una API `identify`.
Presentar **un objeto por foto**, completo, sobre fondo limpio. No requiere entrenamiento.

1. En **Entorno de ejecución → Cambiar tipo de entorno**, seleccionar GPU si está disponible; CPU también sirve para probar, con mayor latencia.
2. Ejecutar las celdas 1–5 en orden. La carga inicial descarga el modelo.
3. Ejecutar la celda 6, introducir una contraseña y abrir en el celular el enlace `gradio.live` que aparezca.
4. Para mejorar las comparaciones, añadir fotos reales con la celda opcional de referencias.

El enlace es temporal: desaparece si Colab termina la sesión. Colab gratuito restringe interactuar principalmente por una interfaz web externa; usar este notebook para pruebas interactivas y revisar las condiciones de Colab. Un plan de pago con saldo no garantiza disponibilidad permanente.

Hay **11 candidatos** en la lista actual; el reto exige elegir **9**. Antes de integrar LEDs, ajustar la lista y definir sus ubicaciones reales.

Fuentes: [Google: modelo y ejemplos](https://ai.google.dev/gemma/docs/embeddinggemma/multimodal-embeddinggemma-with-sentence-transformers), [Gradio: enlaces y API](https://gradio.app/guides/sharing-your-app), [Colab: condiciones](https://research.google.com/colaboratory/faq.html).

## 1. Instalar dependencias
Si Colab pide reiniciar la sesión después de instalar, reiniciar y continuar con la celda 2. No reinstalamos PyTorch: usamos el de Colab.

In [ ]:
%pip install -q -U sentence-transformers transformers gradio


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configurar categorías y cargar el modelo
Las descripciones de cable, capacitores y soldadura deben ajustarse a los objetos reales. Los IDs son provisionales; no representan todavía posiciones físicas.

`MIN_SIMILARITY` y `MIN_MARGIN` quedan sin configurar: inicialmente veremos candidatos, pero `recognized` será falso. Ajustarlos solo después de pruebas con objetos conocidos, desconocidos y la mesa vacía. La similitud **no es un porcentaje de confianza**.

In [ ]:
import os
os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
import time
import numpy as np
import torch
from PIL import Image, ImageOps
from sentence_transformers import SentenceTransformer

MODEL_ID = 'google/embeddinggemma-2'
CATEGORIES = {
    'soldering_station': 'An electronics soldering and hot air rework station with a control unit and iron holder',
    'digital_multimeter': 'A handheld digital multimeter with a screen and rotary selector',
    'bench_power_supply': 'A laboratory bench DC power supply with output terminals and controls',
    'servo_motor': 'A small blue micro servo motor with a three-wire cable',
    'oscilloscope': 'A digital oscilloscope with a waveform screen and input connectors',
    'function_generator': 'A laboratory function generator with frequency controls and signal outputs',
    'variable_capacitor': 'A laboratory variable parallel plate capacitor with circular metal plates on a base',
    'capacitors': 'A small cylindrical electrolytic capacitor with two wire leads',
    'power_adapter': 'A rectangular AC DC power adapter with an integrated output cable',
    'multimeter_box': 'A yellow orange cardboard retail box for an AstroAI digital multimeter',
    'solder': 'An electric soldering iron with a handle and pointed metal tip',
    'otros': 'An everyday object that is not one of the laboratory tools, such as household items, hand tools or bare circuit boards',
}
MIN_SIMILARITY = None
MIN_MARGIN = None
LOCATION_BY_CATEGORY = {}  # Ejemplo al definir posiciones reales: {'digital_multimeter': 2}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float32
print('Dispositivo:', device, '| precisión:', dtype)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('Cargando el modelo; la primera vez puede tardar varios minutos...')
model = SentenceTransformer(
    MODEL_ID,
    device=device,
    config_kwargs={'audio_config': None},
    model_kwargs={'torch_dtype': dtype},
)
model.eval()
print('Modelo cargado:', MODEL_ID)
print('Categorías:', len(CATEGORIES), '(seleccionar 9 antes de integrar los LEDs)')

## 3. Decidir cuándo aceptar una comparación
Rechazamos resultados débiles o ambiguos. Si faltan umbrales de validación, mostramos el candidato sin autorizar la señalización.

In [ ]:
import math

def decide(scores, minimum=None, margin=None):
    ranking = sorted(
        ((name, float(score)) for name, score in scores.items() if math.isfinite(float(score))),
        key=lambda item: item[1], reverse=True,
    )
    result = {'candidate': None, 'recognized': False, 'similarity': None,
              'margin': None, 'reason': 'No hay suficientes categorías válidas.'}
    if not ranking:
        return result
    result.update(candidate=ranking[0][0], similarity=ranking[0][1])
    if len(ranking) < 2:
        return result
    result['margin'] = ranking[0][1] - ranking[1][1]
    if minimum is None or margin is None:
        result['reason'] = 'Modo de prueba: umbrales sin calibrar.'
    elif ranking[0][1] < minimum:
        result['reason'] = 'Similitud insuficiente: objeto desconocido o captura inadecuada.'
    elif result['margin'] < margin:
        result['reason'] = 'Comparación ambigua entre las categorías principales.'
    else:
        result.update(recognized=True, reason='Comparación aceptada con los umbrales configurados.')
    return result

## 4. Preparar la comparación y la respuesta de la API
El modo inicial compara la imagen con las descripciones. Si cargamos referencias en la celda 5, compara con fotografías reales. Cambiar de modo requiere volver a validar los umbrales.

In [ ]:
names = list(CATEGORIES)
text_vectors = model.encode(list(CATEGORIES.values()), normalize_embeddings=True, convert_to_numpy=True)
reference_vectors = None
reference_labels = []

def prepare_image(image):
    image = ImageOps.exif_transpose(image).convert('RGB')
    image.thumbnail((1024, 1024))
    return image

def identify(image):
    if image is None:
        return {'recognized': False, 'candidate': None, 'location_id': None,
                'reason': 'Sube una fotografía.'}
    start = time.perf_counter()
    vector = model.encode({'image': prepare_image(image)}, normalize_embeddings=True, convert_to_numpy=True)
    if reference_vectors is None:
        raw_scores = np.asarray(text_vectors @ vector).reshape(-1)
        scores = dict(zip(names, raw_scores))
        mode = 'image_to_text'
    else:
        raw_scores = np.asarray(reference_vectors @ vector).reshape(-1)
        scores = {name: max(float(score) for label, score in zip(reference_labels, raw_scores) if label == name)
                  for name in names}
        mode = 'image_to_image'
    result = decide(scores, MIN_SIMILARITY, MIN_MARGIN)
    result['mode'] = mode
    result['location_id'] = LOCATION_BY_CATEGORY.get(result['candidate']) if result['recognized'] else None
    result['elapsed_seconds'] = round(time.perf_counter() - start, 3)
    result['ranking'] = [{'object': name, 'similarity': round(float(score), 4)}
                         for name, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)]
    return result

print('Comparación preparada. Las referencias y los umbrales se pueden ajustar sin recargar el modelo.')

## 5. Cargar fotos de referencia desde Drive

Guardar `dataset_prepared/references.zip` en `Mi unidad/Kiwi/references.zip`. Esta celda lee el archivo desde Drive y calcula los vectores de las 41 referencias. Mantener `LOAD_REFERENCE_ZIP = True` para activar la comparación imagen contra imagen. Las 3 fotos de `evaluation.zip` se reservan para pruebas.

Los vectores quedan en memoria durante la sesión; este notebook todavía no genera ni guarda `reference_embeddings.npz`. Al cambiar el ZIP hay que ejecutar de nuevo esta celda. Volver a ejecutar la sección 4 limpia las referencias anteriores.


In [ ]:
LOAD_REFERENCE_ZIP = True
if LOAD_REFERENCE_ZIP:
    import io
    import zipfile
    from pathlib import PurePosixPath, Path
    zip_path = Path('/content/drive/MyDrive/Kiwi/references.zip')
    if not zip_path.is_file():
        available = [p.name for p in zip_path.parent.glob('*.zip')]
        raise FileNotFoundError(f'No existe {zip_path}. ZIPs disponibles en Kiwi: {available}')
    zip_name, zip_data = zip_path.name, zip_path.read_bytes()
    print('Cargando referencias desde Drive:', zip_name)
    if len(zip_data) > 100 * 1024 * 1024:
        raise ValueError('El ZIP debe pesar menos de 100 MB.')
    vectors, labels = [], []
    with zipfile.ZipFile(io.BytesIO(zip_data)) as archive:
        members = archive.infolist()
        if sum(member.file_size for member in members) > 500 * 1024 * 1024:
            raise ValueError('El contenido del ZIP supera 500 MB.')
        for member in members:
            path = PurePosixPath(member.filename)
            if member.is_dir() or path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}:
                continue
            if '__MACOSX' in path.parts or path.name.startswith('.'):
                continue
            label = path.parent.name
            if label not in CATEGORIES:
                raise ValueError(f'Carpeta desconocida: {label}. Usar los nombres de CATEGORIES.')
            if member.file_size > 15 * 1024 * 1024:
                raise ValueError(f'Foto demasiado grande: {path.name}. Máximo 15 MB.')
            with Image.open(io.BytesIO(archive.read(member))) as photo:
                vector = model.encode({'image': prepare_image(photo)}, normalize_embeddings=True, convert_to_numpy=True)
            vectors.append(vector)
            labels.append(label)
    missing = set(CATEGORIES) - set(labels)
    if missing:
        raise ValueError('Faltan referencias de: ' + ', '.join(sorted(missing)))
    reference_vectors = np.stack(vectors)
    reference_labels = labels
    MIN_SIMILARITY = None
    MIN_MARGIN = None
    print('Referencias cargadas:', {name: labels.count(name) for name in names})
    print('Se restablecieron los umbrales: calibrar para el nuevo modo.')
else:
    print('Sin cambios en las referencias. Modo actual:', 'fotografías' if reference_vectors is not None else 'descripciones')

## 5b. Reconocimiento con un VLM, sin fotos de referencia (recomendado)
**Qwen3-VL-4B-Instruct** es un modelo de visión y lenguaje que ya sabe qué es un multímetro, una resistencia o un Arduino. Recibe la foto y una pregunta de opción múltiple con las descripciones de `CATEGORIES`; se leen las probabilidades de cada respuesta. No necesita `references.zip`, y los objetos ajenos caen en la opción *ninguna* (`otros`).

- `similarity` pasa a ser una **probabilidad** (0–1) y `margin` la ventaja sobre la segunda opción. 
- Reemplaza `identify`: la API REST (6b) y Gradio (6) lo usan sin volver a ejecutarlas. Para volver a las fotos de referencia: `identify = identify_embeddings`.
- **Doble verificación:** si la herramienta tiene fotos de referencia (sección 5) y EmbeddingGemma elige otra, no se acepta. Así una foto ambigua (p. ej. un equipo visto por detrás) no enciende un LED equivocado.
- Aceptación: probabilidad ≥ 0,90 y margen ≥ 0,50. En la Pi dejar `decision.min_similarity` y `min_margin` en `null` para usar esta decisión.
- Ocupa ~9 GB de GPU (cabe en una T4 junto a EmbeddingGemma) y tarda ~1–2 s por foto.
- Si se vuelve a ejecutar la sección 4, hay que ejecutar otra vez esta celda.

In [ ]:
# Kiwi · reconocimiento con un modelo de visión y lenguaje (VLM), sin fotos de referencia.
#
# Ejecutar después de las secciones 1–5 (usa CATEGORIES, prepare_image e identify).
# Reemplaza `identify`, así que la API REST de la sección 6b y la página de Gradio pasan a usar el VLM
# sin volver a ejecutarlas. Para volver a EmbeddingGemma: identify = identify_embeddings
#
# El VLM recibe la foto y una pregunta de opción múltiple con las categorías; se leen las probabilidades
# de cada letra de respuesta. `similarity` pasa a ser esa probabilidad (0–1) y `margin` la diferencia con
# la segunda opción. La opción "ninguna" se devuelve como la categoría `otros`.
import time
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

VLM_ID = 'Qwen/Qwen3-VL-4B-Instruct'
VLM_MIN_PROBABILITY = 0.90   # recognized=True solo por encima de esta probabilidad...
VLM_MIN_MARGIN = 0.50        # ...y con esta ventaja sobre la segunda opción
# Doble verificación: si la categoría tiene fotos de referencia (sección 5) y EmbeddingGemma elige otra
# herramienta, no se acepta. Evita LEDs equivocados con fotos ambiguas (p. ej. equipos vistos por detrás).
VLM_CHECK_WITH_REFERENCES = True
OTHER = 'otros'

if 'identify_embeddings' not in globals():
    identify_embeddings = identify   # el reconocimiento por fotos de referencia sigue disponible

if 'vlm_model' not in globals():
    print('Cargando', VLM_ID, '(la primera vez descarga ~9 GB)...')
    vlm_processor = AutoProcessor.from_pretrained(
        VLM_ID, min_pixels=256 * 28 * 28, max_pixels=640 * 28 * 28)  # fotos de ~0,5 MP: rápido y suficiente
    vlm_model = AutoModelForImageTextToText.from_pretrained(
        VLM_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto', attn_implementation='sdpa')
    vlm_model.eval()
    print('VLM cargado. Memoria GPU usada: %.1f GB' % (torch.cuda.memory_allocated() / 1e9))

# Descripciones para el VLM: rasgos del panel frontal que distinguen equipos parecidos por fuera.
# Las categorías sin entrada aquí usan su texto de CATEGORIES.
VLM_DESCRIPTIONS = {
    'oscilloscope': 'Oscilloscope: bench instrument whose front has a large screen with a grid showing waveforms, '
                    'many knobs and BNC probe inputs labeled CH1/CH2',
    'function_generator': 'Function generator: bench instrument whose front has a numeric frequency display, '
                          'waveform buttons (sine, square, triangle) and BNC signal output connectors',
    'bench_power_supply': 'Bench DC power supply: bench instrument whose front shows voltage (V) and current (A) '
                          'readouts, adjustment knobs and red/black banana output terminals',
    'soldering_station': 'Soldering station: control unit with a temperature display or dial, connected to a '
                         'soldering iron and an iron holder stand',
    'power_adapter': 'AC/DC power adapter: small black plastic brick with an attached cable, used to power a '
                     'device from the wall outlet',
    'digital_multimeter': 'Handheld digital multimeter: portable meter with an LCD screen, a large rotary selector '
                          'dial and probe jacks (COM, V, A)',
    'solder': 'Soldering iron: pen-shaped hand tool with a plastic handle and a long pointed metal tip',
}

VLM_OPTIONS = [name for name in CATEGORIES if name != OTHER] + [OTHER]
VLM_LETTERS = [chr(ord('A') + i) for i in range(len(VLM_OPTIONS))]
VLM_LETTER_IDS = [vlm_processor.tokenizer.encode(letter, add_special_tokens=False)[0] for letter in VLM_LETTERS]
assert len(set(VLM_LETTER_IDS)) == len(VLM_LETTERS), 'Cada letra debe ser un token distinto.'


def vlm_prompt():
    lines = [
        'You are checking which tool someone is returning to an electronics lab shelf.',
        'Look only at the main object in the photo; ignore hands, tables, cables in the background and other clutter.',
        'Many bench instruments are similar boxes: decide by what is visible on their front panel. If only the back, '
        'bottom or side of an instrument is visible and you cannot see its front panel, choose the last option.',
        'Options:',
    ]
    for letter, name in zip(VLM_LETTERS, VLM_OPTIONS):
        if name == OTHER:
            lines.append(f'{letter}. None of the above or cannot tell: the main object is something else (for '
                         'example a resistor, LED, loose wire, Arduino or other circuit board, pliers, screwdriver, '
                         'phone or household item), there is no clear object, or only the back/side of an '
                         'instrument is visible.')
        else:
            lines.append(f'{letter}. {VLM_DESCRIPTIONS.get(name, CATEGORIES[name])}.')
    lines.append('Answer with only the letter of the correct option.')
    return '\n'.join(lines)


@torch.inference_mode()
def vlm_scores(image):
    messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image},
                                             {'type': 'text', 'text': vlm_prompt()}]}]
    inputs = vlm_processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                               return_dict=True, return_tensors='pt').to(vlm_model.device)
    logits = vlm_model(**inputs).logits[0, -1].float()
    probabilities = torch.softmax(logits[VLM_LETTER_IDS], dim=0).tolist()
    return dict(zip(VLM_OPTIONS, probabilities))


def identify_vlm(image):
    if image is None:
        return {'recognized': False, 'candidate': None, 'location_id': None, 'reason': 'Sube una fotografía.'}
    start = time.perf_counter()
    scores = vlm_scores(prepare_image(image))
    ranking = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    candidate, probability = ranking[0]
    margin = probability - ranking[1][1]
    result = {'candidate': candidate, 'similarity': probability, 'margin': margin, 'mode': 'vlm',
              'model': VLM_ID, 'recognized': False}
    if candidate == OTHER:
        result['reason'] = 'No es una de las herramientas o no se ve su frente. Muestra el frente del objeto.'
    elif probability < VLM_MIN_PROBABILITY:
        result['reason'] = 'Confianza insuficiente: foto poco clara u objeto desconocido.'
    elif margin < VLM_MIN_MARGIN:
        result['reason'] = 'Duda entre dos herramientas. Repite la foto.'
    else:
        result.update(recognized=True, reason='Reconocido por el VLM.')
        if (VLM_CHECK_WITH_REFERENCES and globals().get('reference_vectors') is not None
                and candidate in set(reference_labels)):
            check = identify_embeddings(image)
            result['reference_candidate'] = check['candidate']
            result['reference_similarity'] = check['similarity']
            if check['candidate'] != candidate:
                result.update(recognized=False, reason='El VLM y las fotos de referencia no coinciden. '
                                                        'Muestra el frente de la herramienta.')
    result['location_id'] = LOCATION_BY_CATEGORY.get(candidate) if result['recognized'] else None
    result['elapsed_seconds'] = round(time.perf_counter() - start, 3)
    result['ranking'] = [{'object': name, 'similarity': round(p, 4)} for name, p in ranking]
    return result


identify = identify_vlm
_warm = identify_vlm(Image.new('RGB', (448, 448), 'white'))   # primera inferencia (compila kernels)
print('Reconocimiento activo: VLM', VLM_ID, '| opciones:', ', '.join(f'{l}={n}' for l, n in zip(VLM_LETTERS, VLM_OPTIONS)))
print('Prueba con imagen en blanco ->', _warm['candidate'], round(_warm['similarity'], 3), f"({_warm['elapsed_seconds']} s)")


## 6. Abrir la página del celular y la API
Ejecutar esta celda e introducir una contraseña para esta sesión. El usuario será `kiwi`.
Abrir el enlace **https://…gradio.live** desde el celular y seleccionar una foto de la galería o usar la cámara cuando el navegador lo permita. No requiere que el celular y la computadora estén en la misma red.

El acceso es temporal y pasa por el servicio de enlaces de Gradio. No usar fotos sensibles. Mantener la sesión de Colab activa; cerrar el servicio con `demo.close()` al terminar.

La página devuelve candidatos y similitudes. Al inicio `recognized=false` es intencional: no se ha validado la aceptación. No hay LEDs conectados en este notebook.

In [ ]:
import getpass
import gradio as gr

if 'demo' in globals():
    demo.close()
APP_USER = 'kiwi'
APP_PASSWORD = getpass.getpass('Contraseña para la página y la API (mínimo 12 caracteres): ')
if len(APP_PASSWORD) < 12:
    raise ValueError('Usa al menos 12 caracteres y vuelve a ejecutar esta celda.')
demo = gr.Interface(
    fn=identify,
    inputs=gr.Image(type='pil', sources=['upload', 'webcam'], label='Fotografía de una herramienta'),
    outputs=gr.JSON(label='Resultado'),
    title='Kiwi · ¿Dónde guardo esta herramienta?',
    description='Presenta un objeto por foto. El resultado inicial es un candidato de prueba; no es una certeza.',
    api_name='identify',
    flagging_mode='never',
    submit_btn='Identificar',
    clear_btn='Limpiar',
)
demo.queue(default_concurrency_limit=1, max_size=10)
demo.launch(share=True, auth=(APP_USER, APP_PASSWORD), debug=False)

## 6b. API REST para la Raspberry Pi (ngrok)
La estación de la Raspberry Pi (`pi/`) envía cada foto a `POST /predict` y consulta `GET /health` para mostrar **API en línea**. Esta celda publica esas dos rutas con FastAPI y un túnel de ngrok. Puede funcionar junto con la página de Gradio de la sección 6 o sin ella.

Antes de ejecutarla, en **Secretos** (icono de llave) de Colab crear y habilitar para este notebook:
- `NGROK_AUTHTOKEN`: token de una cuenta gratuita de ngrok.
- `KIWI_API_KEY`: una clave larga inventada por el equipo. La misma va en la Pi como variable `KIWI_API_KEY`.

Copiar la URL que termina en `/predict` en el campo **Endpoint de Colab** de la pantalla táctil (o en `pi/config.yaml`). La URL cambia en cada ejecución salvo que se use un dominio fijo de ngrok.

In [ ]:
%pip install -q pyngrok python-multipart

In [ ]:
import io
import secrets
import threading
import uvicorn
from fastapi import FastAPI, File, Header, HTTPException, UploadFile
from google.colab import userdata
from pyngrok import ngrok

API_KEY = userdata.get('KIWI_API_KEY')
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')
if not API_KEY or len(API_KEY) < 16:
    raise ValueError('Crea el secreto KIWI_API_KEY con al menos 16 caracteres.')
API_PORT = 8008
model_lock = threading.Lock()
api = FastAPI(title='Kiwi API')

@api.get('/health')
def health():
    return {'ok': True,
            'mode': 'image_to_image' if reference_vectors is not None else 'image_to_text',
            'calibrated': MIN_SIMILARITY is not None and MIN_MARGIN is not None,
            'categories': names}

@api.post('/predict')
def predict(file: UploadFile = File(...), authorization: str | None = Header(None)):
    if not secrets.compare_digest(authorization or '', f'Bearer {API_KEY}'):
        raise HTTPException(401, 'Clave incorrecta.')
    data = file.file.read(15 * 1024 * 1024 + 1)
    if len(data) > 15 * 1024 * 1024:
        raise HTTPException(413, 'La foto supera 15 MB.')
    try:
        image = Image.open(io.BytesIO(data))
        image.load()
    except Exception:
        raise HTTPException(400, 'El archivo no es una imagen válida.')
    with model_lock:  # una inferencia a la vez en la GPU
        return identify(image)

if 'api_server' in globals():
    api_server.should_exit = True
    ngrok.kill()
api_server = uvicorn.Server(uvicorn.Config(api, host='127.0.0.1', port=API_PORT, log_level='warning'))
threading.Thread(target=api_server.run, daemon=True).start()
ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(API_PORT, 'http')
print('Endpoint para la Raspberry Pi:', tunnel.public_url + '/predict')
print('Modo:', health()['mode'], '| umbrales calibrados:', health()['calibrated'])

## 7. Consumir la API desde otra computadora o aplicación
La estación de la Raspberry Pi (`pi/`) usa la API REST de la sección 6b. También puede usar el enlace de Gradio con `recognizer.driver: gradio`.

La página del celular ya envía la foto a la API. No hace falta escribir un cliente para probarla.
Para una integración propia, usar **Use via API** en el pie de la página y consultar el esquema generado. Gradio utiliza su protocolo de carga y cola; `identify` no es una ruta REST simple para enviar JPEG directamente.

Ejemplo de cliente Python (ejecutar en la computadora cliente):
```python
from getpass import getpass
from gradio_client import Client, handle_file

client = Client('https://TU-ENLACE.gradio.live', auth=('kiwi', getpass('Contraseña: ')))
result = client.predict(handle_file('foto.jpg'), api_name='/identify')
print(result)
```
Instalar `gradio_client` en el cliente. No guardar contraseñas en el notebook ni en código compartido.

## Validación antes de conectar luces
- Elegir las **9 categorías** definitivas y modificar `CATEGORIES`; ejecutar otra vez desde la celda 4.
- Probar fotografías nuevas, mesa vacía y objetos ajenos. Registrar candidatos, similitudes y confusiones.
- Ajustar `MIN_SIMILARITY` y `MIN_MARGIN` para el modo que se utilizará, ejecutando la celda 2 únicamente si se quiere recargar todo. También se pueden asignar estas variables en una nueva celda sin recargar el modelo.
- Definir `LOCATION_BY_CATEGORY` con las ubicaciones reales.
- El cliente debe activar luces solo cuando `recognized` sea verdadero y `location_id` tenga un valor válido. Ante rechazo o error, apagar la señalización anterior.

## Problemas frecuentes
- **Modelo desconocido:** ejecutar instalación; reiniciar sesión si quedaron librerías antiguas en memoria.
- **Acceso al modelo denegado:** revisar la página oficial en Hugging Face. Si requiere autenticación, usar un secreto `HF_TOKEN` de Colab; no pegar tokens en celdas compartidas.
- **Sin memoria GPU:** reducir el tamaño de las fotos o usar CPU; no cambiar a `float16`, que el modelo no admite.
- **Enlace dejó de funcionar:** comprobar que la sesión siga conectada y volver a ejecutar la celda 6. El enlace puede cambiar.
- **Dos equipos se confunden:** mejorar encuadre y fotos de referencia; no activar LEDs con una diferencia pequeña entre candidatos.